Initially I did balancing of the datasets because there is some minor class imbalance, but after training with and without the balanced data this turned out to not be necessary.

In [ ]:
def balance_dataset(dataset):
    images_list = []
    labels_list = []
    
    for img, label in dataset:
        images_list.append(img)
        labels_list.append(label)
    
    images_list = np.array(images_list)
    labels_list = np.array(labels_list)
    
    # Get unique labels and their counts
    unique_labels, counts = np.unique(labels_list, return_counts=True)
    
    # Get the size of the majority class
    target_size = max(counts)
    
    # Create balanced dataset
    balanced_images = []
    balanced_labels = []
    
    for label, count in zip(unique_labels, counts):
        # Get indices for current label
        indices = np.where(labels_list == label)[0]
        label_images = images_list[indices]
        
        # Calculate how many augmentations needed
        num_augmentations = target_size - count
        
        if num_augmentations > 0:
            # Randomly select images to augment
            selected_indices = np.random.choice(len(indices), size=num_augmentations, replace=True)
            selected_images = label_images[selected_indices]
            
            # Augment selected images using your existing simclr_augment function
            for img in selected_images:
                augmented_img = simclr_augment(img)
                balanced_images.append(augmented_img)
                balanced_labels.append(label)
        
        # Add original images
        balanced_images.extend(label_images)
        balanced_labels.extend([label] * len(label_images))
    
    # Convert back to tensors
    balanced_images = tf.stack(balanced_images)
    balanced_labels = tf.convert_to_tensor(balanced_labels)
    
    # Create new dataset
    balanced_dataset = tf.data.Dataset.from_tensor_slices((balanced_images, balanced_labels))
    return balanced_dataset

# Balance the training dataset
balanced_train_ds = balance_dataset(train_ds)
balanced_val_ds = balance_dataset(val_ds)

def print_label_distribution(dataset, dataset_name="Dataset"):
    # Get the labels from the dataset
    labels = np.array([label.numpy() for _, label in dataset])
    
    # Count the occurrences of each label
    unique_labels, counts = np.unique(labels, return_counts=True)
    
    # Print the distribution
    print(f"\nDistribution of Labels in {dataset_name}:")
    for label, count in zip(unique_labels, counts):
        print(f"Label {label}: {count}")

# Print distributions for training and validation sets
print_label_distribution(balanced_train_ds, "Balanced Training Dataset")
print_label_distribution(balanced_val_ds, "Balanced Validation Dataset")